# FitBit Activity & Health Data — EDA
**Dataset:** FitBit_data.csv — 457 records, 33 unique users, 15 features  
**Business Goal:** Understand user activity patterns, sedentary behavior, and health metric correlations  
**Domain:** Health Analytics / Wearable Device Data


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 130, 'axes.titlesize': 14, 'axes.titleweight': 'bold'})

## 1. Load & Inspect

In [ ]:
df = pd.read_csv('../data/FitBit_data.csv')
print('Shape:', df.shape)
print('Unique users:', df['Id'].nunique())
print('Missing values:', df.isnull().sum().sum())
df.head(3)

## 2. Feature Engineering

In [ ]:
df['ActivityDate'] = pd.to_datetime(df['ActivityDate'])
df['DayOfWeek'] = df['ActivityDate'].dt.day_name()

df['TotalActiveMinutes'] = (
    df['VeryActiveMinutes'] +
    df['FairlyActiveMinutes'] +
    df['LightlyActiveMinutes']
)
df['SedentaryHours'] = df['SedentaryMinutes'] / 60

df['ActivityLevel'] = pd.cut(
    df['TotalSteps'],
    bins=[0, 5000, 10000, 15000, np.inf],
    labels=['Sedentary (<5k)', 'Low Active (5k-10k)', 'Active (10k-15k)', 'Very Active (>15k)']
)

print('New features created: DayOfWeek, TotalActiveMinutes, SedentaryHours, ActivityLevel')
df[['TotalSteps', 'TotalActiveMinutes', 'SedentaryHours', 'ActivityLevel']].describe()

## 3. Key Stats Summary

In [ ]:
print(f"Avg Daily Steps       : {df['TotalSteps'].mean():,.0f}")
print(f"Days Meeting 10k Goal : {(df['TotalSteps'] >= 10000).mean()*100:.1f}%")
print(f"Avg Calories/Day      : {df['Calories'].mean():,.0f}")
print(f"Avg Sedentary Hrs/Day : {df['SedentaryHours'].mean():.1f}h")
print(f"Steps-Calories Corr   : {df['TotalSteps'].corr(df['Calories']):.2f}")

## 4. Step Count Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df['TotalSteps'], bins=40, kde=True, color='#1565C0', ax=ax)
ax.axvline(10000, color='red', linestyle='--', linewidth=1.5, label='10k step goal')
ax.axvline(df['TotalSteps'].mean(), color='orange', linestyle='--',
           linewidth=1.5, label=f"Mean: {df['TotalSteps'].mean():,.0f}")
ax.set_title('Daily Step Count Distribution')
ax.set_xlabel('Total Steps')
ax.legend()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('../outputs/01_steps_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Activity Level Breakdown

In [ ]:
al = df['ActivityLevel'].value_counts().reindex(
    ['Sedentary (<5k)', 'Low Active (5k-10k)', 'Active (10k-15k)', 'Very Active (>15k)']
)
colors = ['#E53935', '#FB8C00', '#43A047', '#1565C0']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(al.index, al.values, color=colors)
ax.set_title('Activity Level Distribution (Based on Daily Steps)')
ax.set_ylabel('Number of Days')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():,}\n({bar.get_height()/len(df)*100:.1f}%)',
            ha='center', fontsize=9, fontweight='bold')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('../outputs/02_activity_levels.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Steps vs Calories (Correlation)

In [ ]:
corr = df['TotalSteps'].corr(df['Calories'])

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(df['TotalSteps'], df['Calories'], alpha=0.4, color='#1565C0', s=20)
z = np.polyfit(df['TotalSteps'], df['Calories'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['TotalSteps'].min(), df['TotalSteps'].max(), 200)
ax.plot(x_line, p(x_line), 'r--', linewidth=2, label='Trend')
ax.set_title(f'Steps vs Calories Burned (r = {corr:.2f})')
ax.set_xlabel('Total Steps')
ax.set_ylabel('Calories')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/03_steps_vs_calories.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Steps & Calories by Day of Week

In [ ]:
order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
daily_steps = df.groupby('DayOfWeek')['TotalSteps'].mean().reindex(order)
daily_cal   = df.groupby('DayOfWeek')['Calories'].mean().reindex(order)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(daily_steps.index, daily_steps.values,
            color=sns.color_palette('Blues_r', 7))
axes[0].axhline(10000, color='red', linestyle='--', label='10k goal')
axes[0].set_title('Avg Steps by Day of Week')
axes[0].set_ylabel('Avg Steps')
axes[0].legend()
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha='right')

axes[1].plot(daily_cal.index, daily_cal.values, marker='o',
             linewidth=2.5, color='#E65100', markersize=8)
axes[1].fill_between(range(7), daily_cal.values, alpha=0.15, color='#E65100')
axes[1].set_title('Avg Calories Burned by Day of Week')
axes[1].set_ylabel('Avg Calories')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(order, rotation=30, ha='right')

plt.suptitle('Weekly Activity Patterns — FitBit', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/04_weekly_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Sedentary vs Active Time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time breakdown pie
avg_m = {
    'Very Active': df['VeryActiveMinutes'].mean(),
    'Fairly Active': df['FairlyActiveMinutes'].mean(),
    'Lightly Active': df['LightlyActiveMinutes'].mean(),
    'Sedentary': df['SedentaryMinutes'].mean(),
}
axes[0].pie(avg_m.values(), labels=avg_m.keys(),
            autopct='%1.1f%%', colors=['#1565C0','#42A5F5','#A5D6A7','#FFCCBC'],
            wedgeprops=dict(width=0.6))
axes[0].set_title('Avg Daily Time by Activity Type')

# Sedentary vs active scatter
sc = axes[1].scatter(df['SedentaryHours'], df['TotalActiveMinutes'],
                     c=df['Calories'], cmap='YlOrRd', alpha=0.5, s=25)
plt.colorbar(sc, ax=axes[1], label='Calories')
axes[1].set_title('Sedentary Hours vs Active Minutes')
axes[1].set_xlabel('Sedentary Hours')
axes[1].set_ylabel('Total Active Minutes')

plt.suptitle('Activity & Sedentary Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/05_sedentary_vs_active.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Correlation Heatmap

In [ ]:
num_df = df.select_dtypes(include=np.number).drop(columns=['Id'], errors='ignore')

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(num_df.corr(), annot=True, cmap='Blues', fmt='.2f', linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap — FitBit Health Metrics')
plt.tight_layout()
plt.savefig('../outputs/06_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Key Insights

- **Only ~30% of days met the 10,000 step goal** — majority of users are under the recommended threshold
- **Steps and Calories have strong positive correlation (r ≈ 0.59)** — walking is an effective calorie-burn strategy
- **Average sedentary time is ~16 hours/day** — users are predominantly sedentary despite wearing a fitness tracker
- **Tuesday and Saturday show highest step counts** — midweek effort + weekend activity peak
- **Sedentary minutes negatively correlated with active minutes** — obvious but confirmed by data
- **Very Active minutes avg only ~21 min/day** — users rarely reach high-intensity activity zones
- **Wide variance in user behavior** — top users average 3x the steps of least active users
